# SILAS in a notebook

The session is loaded when the notebook's kernel starts (kernel **SILAS**): every command of the
terminal session (`bt`) is there, e.g. `show_scans()`, `analyse_scan(...)`, `commands()`.
Plots appear below the cells; the viewers (`start_mpccd_viewer`, `start_tt_viewer`,
`view_full_frame`, ...) appear as interactive figures: click on a figure, then use its keys
(`h` for help in the MPCCD and time-tool viewers, `q` stops a viewer).

This file is part of the code (`~/SILAS/notebooks/`) and is replaced when the code is synced:
**save your own notebooks somewhere else** (e.g. `/UserData/<user>/notebooks/`).

In [ ]:
# only needed for another experiment config (or in a kernel without the session):
# from silas.cli.notebook import start
# start(config="config/other.json")

## What is there

In [ ]:
commands()          # every command with its first line of help; help(name) for the full text

In [ ]:
show_scans()        # the scans of the run log

## Examples

Remove the `#` of the line you want to run.

In [ ]:
# scan_info(-1)                                    # the newest scan
# analyse_scan(-1, max_tags=100)                   # quick look at the ROI analysis: the plot appears below
# d = load_scan(-1)                                # its results as pandas tables

In [ ]:
# start_mpccd_viewer(run=1234501, max_tags=300)    # offline MPCCD view of a stored run (interactive figure)
# start_tt_viewer(run=1234501)                     # the time-tool camera, shot by shot

In [ ]:
# ff = load_full_frame(-1)                          # a full-frame reduction (full_frame_job makes it)
# view_full_frame(-1)                               # pump on, off and on / off of a step (keys: arrows, r, m, d, b)

### Scan ROI analysis

In [ ]:
# analyse_scan(-1)                                  # analyse the newest scan
# analyse_scan(1234560, tt=True, bin_ps=0.02)       # analyse a specific scan, with time-tool binning

In [ ]:
# scan_results(-1)                      # the results of the newest scan, as pandas tables
# scan_results(1234560, show=True)      # specific scan, show the plots as well

In [ ]:
## Loading using silas
import matplotlib.pyplot as plt

from silas.offline.scan_analysis import load_h5

# Specific h5 file
data = load_h5("scan_123456.h5")
# or
# data = load_scan(123456)

res = data["results"]

roi = res["rois"][0]      # first ROI

x = res["raw"]["x"]
med = res["raw"]["med"][0, :, 0]
lo = res["raw"]["lo"][0, :, 0]
hi = res["raw"]["hi"][0, :, 0]

plt.errorbar(
    x,
    med,
    yerr=[lo, hi],
    marker="o",
    capsize=2,
)

plt.xlabel("Scan position")
plt.ylabel("ROI signal")
plt.title(roi)
plt.grid(True)
plt.show()

In [ ]:
## loading with silas and time tool binning
import matplotlib.pyplot as plt

from silas.offline.scan_analysis import load_h5

# Specific h5 file
data = load_h5("scan_123456.h5")
# or
# data = load_scan(123456)

tt = data["timetool"]

roi = tt["panels"][0][1][0]     # first ROI column

bins = tt["bins"]

x = bins["t_ps"]

med = bins[roi + "_median"]
lo = med - bins[roi + "_q16"]
hi = bins[roi + "_q84"] - med

plt.errorbar(
    x,
    med,
    yerr=[lo, hi],
    marker="o",
    capsize=2,
)

plt.xlabel("Time (ps)")
plt.ylabel("ROI signal")
plt.title(roi)
plt.grid(True)
plt.show()

In [ ]:
## No dependencies on silas, just h5py
import h5py
import matplotlib.pyplot as plt

with h5py.File("scan_123456.h5", "r") as f:

    raw = f["results"]["raw"]

    x = raw["x"][:]
    med = raw["med"][0, :, 0]
    lo = raw["lo"][0, :, 0]
    hi = raw["hi"][0, :, 0]

plt.errorbar(
    x,
    med,
    yerr=[lo, hi],
    marker="o",
    capsize=2,
)

plt.xlabel("Scan position")
plt.ylabel("ROI signal")
plt.grid(True)
plt.show()

In [ ]:
## No dependencies on silas, just h5py
## and time tool binning
import h5py
import matplotlib.pyplot as plt

with h5py.File("scan_123456.h5", "r") as f:

    bins = f["timetool"]["bins"]

    # choose any ROI column
    roi = "roi_my_roi"

    x = bins["t_ps"][:]

    med = bins[roi + "_median"][:]
    lo = med - bins[roi + "_q16"][:]
    hi = bins[roi + "_q84"][:] - med

plt.errorbar(
    x,
    med,
    yerr=[lo, hi],
    marker="o",
    capsize=2,
)

plt.xlabel("Time (ps)")
plt.ylabel("ROI signal")
plt.grid(True)
plt.show()

### Full-frame analysis

Launch the full-frame reduction as a job in the queue on the cluster (`full_frame_job`); in the
notebook you can then look at the results. A scan step or time-tool bin is chosen by its **row**
in `ff.steps` / `ff.bins` (0 = the first); only the viewer (`view_full_frame(step=...)`) counts
steps from 1.

In [ ]:
full_frame_job(1234501)                             # submits: 16 cores, memory estimated
full_frame_job(1234501, panels=[7, 8], tt=True, bin_ps=0.2)     # options as reduce_full_frame
full_frame_job(1234501, cores=32, mem_gb=96)        # more of the node
full_frame_job(1234501, submit=False)               # only print the qsub command

In [ ]:
## Viewing the full-frame reduction results
view_full_frame(1234501)                           # the reduction of the chosen panels (set_ff_panels)
view_full_frame(1234501, step=10, ref="i0_1")      # where it starts (default: the first reference)
view_full_frame(1234501, ref="none")               # start without a reference
view_full_frame("~/data/fullframe_1234501.h5")     # any file, e.g. a copy at home

In [ ]:
## Opening the full-frame reduction results in code using silas
import matplotlib.pyplot as plt

from silas.offline.fullframe import FullFrame

ff = FullFrame("fullframe_1234501_p78.h5")     # in the session also: ff = load_full_frame(1234501)

# Inspect the scan information
ff.steps                         # pandas table of the scan steps (x = motor value, shot counts, ...)
ff.info                          # reduction metadata (dict)

# The scanned motor: "motor" is its alias (the true name if it has none), "motor_true" the true name
motor = ff.info["scan"]["motor"]
motor_true = ff.info["scan"]["motor_true"]
print("Scan motor:", motor, "=", motor_true)

# Choose a scan step: the ROW of ff.steps (0 = first), not the motor position
step = 10
ff.steps.iloc[step]              # scan position and shot counts of this step

# What to look at: "raw" (keV per pixel per shot) or a reference label such as "i0_1"
# (pixel / i0_1 shot by shot); the labels are in ff.labels
ref = "i0_1"

# The statistic over the shots of the step:
# - median: robust against occasional bad shots and outliers
# - mean: every shot contributes; use it for weak signals (a pixel's median is often exactly 0)
# - q16 / q84: the spread of the shots
stat = "median"

# Pump-on and pump-off images, (panels, rows, cols)
on = ff.image(ref, step=step, state="on", stat=stat)
off = ff.image(ref, step=step, state="off", stat=stat)

# On / off of the medians with lower / upper errors (relative errors added in quadrature)
ratio, ratio_lower, ratio_upper = ff.onoff(ref, step=step)

# On - off: of the medians (with errors) or of the means
# difference, difference_lower, difference_upper = ff.difference(ref, step=step)
# difference, _, _ = ff.difference(ref, step=step, stat="mean")

# Only if the reduction was made with step_reference=True: median pixel / median i0_1 of the step
# step_ref, step_ref_lower, step_ref_upper = ff.step_referenced(ref, step=step, state="on")

# All scan steps at once: (steps, panels, rows, cols) - large, one image per step
all_steps = ff.stack(ref, state="on", stat=stat)

# The panels in their detector positions: one 2-D image (gaps NaN)
on_image = ff.assemble(on)
off_image = ff.assemble(off)
ratio_image = ff.assemble(ratio)

# Error images can be assembled the same way
ratio_lower_image = ff.assemble(ratio_lower)
ratio_upper_image = ff.assemble(ratio_upper)

# Simple three-panel view of one step
fig, ax = plt.subplots(1, 3, figsize=(12, 4))

ax[0].imshow(on_image)
ax[0].set_title("pump on")

ax[1].imshow(off_image)
ax[1].set_title("pump off")

ax[2].imshow(ratio_image, cmap="seismic", vmin=0.8, vmax=1.2)
ax[2].set_title("on / off")

for a in ax:
    a.axis("off")

plt.tight_layout()
plt.show()

In [ ]:
# If time-tool binning was on: ff.bins is the bin table (None otherwise)
if ff.bins is not None:
    i = 3                                         # the ROW of ff.bins; its column "bin" is the bin number
    print(ff.bins.iloc[i][["bin", "t_ps", "n", "computed"]])   # computed 0: too few shots, the images are NaN

    bin_on = ff.image(ref, bin=i)                 # pump on (no state for bins), pixel / i0_1, median
    bin_onoff = ff.image("onoff_" + ref, bin=i)   # every shot / the median pump-off image of its step
    bin_diff = ff.image("diff_" + ref, bin=i)     # every shot - the median pump-off image of its step
    t_ps = ff.bins["t_ps"].iloc[i]                # time of this bin (ps)

    plt.imshow(ff.assemble(bin_onoff), cmap="seismic", vmin=0.8, vmax=1.2)
    plt.title("on / off, bin at {:.3f} ps".format(t_ps))
    plt.show()

ff.close()

In [ ]:
## Opening the full-frame reduction results in code without silas (h5py, numpy, matplotlib)
import json
import math

import h5py
import matplotlib.pyplot as plt
import numpy as np


def assemble(panels, info):
    """(panels, rows, cols) -> one 2-D image in detector geometry, as ff.assemble: detector origin
    in the centre pixel, X to the right, Y up, nearest panel pixel, NaN between the panels.
    info: json.loads(f.attrs["info"]) of the full-frame file."""
    geo, det = info["geometry"], info["detector_info"]
    offsets = geo.get("panel_offsets") or {}
    rnd = {"floor": np.floor, "trunc": np.trunc, "rint": np.rint}[geo.get("rounding", "floor")]
    sub_r, sub_c = geo.get("sub_pixel", (0.0, 0.0))
    shift = float(det[0].get("mp_manipulator_shift", 0.0))          # aperture opening (um)
    n, h, w = np.shape(panels)
    pos = []                                                          # (x0, y0, angle, pixel) per panel
    for k, d in enumerate(det):
        off = offsets.get(str(k + 1)) or {}
        a = math.radians(float(d.get("mp_stagedirection", 0.0)))
        s = float(d.get("mp_shiftweight", 0.0)) * shift
        pix = float(d.get("mp_pixelsizex", d.get("mp_pixelsize_x", 50.0)))
        pos.append((float(d["mp_posx"]) + float(off.get("dx_um", 0.0)) + math.cos(a) * s,
                    float(d["mp_posy"]) + float(off.get("dy_um", 0.0)) + math.sin(a) * s,
                    math.radians(float(d["mp_rotationangle"]) + float(off.get("drot_deg", 0.0))), pix))

    def corners(x0, y0, th, pix):                                     # X, Y (um) of the corner pixel centres
        u = np.array([0.0, w - 1, 0.0, w - 1])
        v = -np.array([0.0, 0.0, h - 1, h - 1])
        return (x0 + pix * (u * math.cos(th) - v * math.sin(th)),
                y0 + pix * (u * math.sin(th) + v * math.cos(th)))

    pix0 = pos[0][3]
    cs = [corners(*p) for p in pos]
    half_r = int(math.ceil(max(np.abs(y).max() for _, y in cs) / pix0)) + 2
    half_c = int(math.ceil(max(np.abs(x).max() for x, _ in cs) / pix0)) + 2
    image = np.full((2 * half_r + 1, 2 * half_c + 1), np.nan, np.float32)
    for k, ((x0, y0, th, pix), (x, y)) in enumerate(zip(pos, cs)):
        c_lo = max(int(math.floor(x.min() / pix0)) + half_c - 1, 0)
        c_hi = min(int(math.ceil(x.max() / pix0)) + half_c + 1, image.shape[1] - 1)
        r_lo = max(int(math.floor(-y.max() / pix0)) + half_r - 1, 0)
        r_hi = min(int(math.ceil(-y.min() / pix0)) + half_r + 1, image.shape[0] - 1)
        rr, cc = np.mgrid[r_lo:r_hi + 1, c_lo:c_hi + 1]
        dx = ((cc - half_c) * pix0 - x0) / pix                        # image pixel -> panel pixel
        dy = (-(rr - half_r) * pix0 - y0) / pix
        col = rnd(dx * math.cos(th) + dy * math.sin(th) + sub_c).astype(int)
        row = rnd(dx * math.sin(th) - dy * math.cos(th) + sub_r).astype(int)
        ok = (row >= 0) & (row < h) & (col >= 0) & (col < w)
        image[rr[ok], cc[ok]] = np.asarray(panels[k], np.float32)[row[ok], col[ok]]
    return image


step, ref, stat = 10, "i0_1", "median"

with h5py.File("fullframe_1234501_p78.h5", "r") as f:
    info = json.loads(f.attrs["info"])
    states = info["states"]                       # ["on", "off"] (["all"] without a pump detector)
    stats = info["stats"]                         # ["q16", "median", "q84", "mean"]

    # steps/raw: (step, state, stat, panel, row, col); steps/per_<ref>: pixel / ref per shot
    images = f["steps/per_" + ref]                # or f["steps/raw"]
    on = images[step, states.index("on"), stats.index(stat)]      # (panels, rows, cols)
    off = images[step, states.index("off"), stats.index(stat)]
    x = f["step_table/x"][step]                   # motor value of this step

with np.errstate(invalid="ignore", divide="ignore"):
    ratio = on / off                              # on / off of the medians (FullFrame.onoff adds errors)

# Assemble into detector geometry (the same image as ff.assemble)
on_image = assemble(on, info)
off_image = assemble(off, info)
ratio_image = assemble(ratio, info)

fig, ax = plt.subplots(1, 3, figsize=(12, 4))

ax[0].imshow(on_image)
ax[0].set_title("pump on")

ax[1].imshow(off_image)
ax[1].set_title("pump off")

ax[2].imshow(ratio_image, cmap="seismic", vmin=0.8, vmax=1.2)
ax[2].set_title("on / off")

for a in ax:
    a.axis("off")

plt.tight_layout()
plt.show()

In [ ]:
# If time-tool binning was on (without silas; assemble() from the cell above)
with h5py.File("fullframe_1234501_p78.h5", "r") as f:
    if "bin_table" in f:
        i = 3                                                    # the row of bin_table
        t_ps = f["bin_table/t_ps"][i]                            # time of this bin (ps)
        # bins/...: (bin, stat, panel, row, col), pump on only
        bin_on = f["bins/per_" + ref][i, stats.index(stat)]
        bin_onoff = f["bins/onoff_per_" + ref][i, stats.index(stat)]   # shot / median off image of its step
        bin_image = assemble(bin_onoff, info)

#### Point detectors

Every point detector of the config (except the X-ray shutter, the pump detector and the scanned
motor) is stored for every shot, in the rows of the shot table, and summarised per step and
pump state. Labels: the alias (else the true name with `/` replaced by `_`).

In [ ]:
# Using silas
from silas.offline.fullframe import FullFrame

ff = FullFrame("fullframe_1234501_p78.h5")

ff.point_info                    # {label: {"detector": true name, "alias": ..., "unit": ...}}
ff.points.columns                # the labels

# One detector, every shot (the rows of ff.shots; NaN where not recorded)
i0 = ff.points["i0_1"]

# Several detectors
points = ff.points[["i0_1", "i0_2"]]

# Together with the shot table (run, tag, step_index, state, read, ...)
shots = ff.shots.join(ff.points)
used = shots[shots["read"].astype(bool)]           # only the shots that are in the images

# Mean i0_1 per scan step and pump state (state 0 = on, 1 = off)
used.groupby(["step_index", "state"])["i0_1"].mean()

# The same per step, already computed: <label>_<state>_n / _mean / _std / _q16 / _median / _q84
ff.point_steps[["i0_1_on_mean", "i0_1_off_mean"]]

# Time-tool bins: the shots' "bin" is the bin number of ff.bins["bin"]
if ff.bins is not None:
    used[used["bin"] == ff.bins["bin"].iloc[3]]["i0_1"].mean()

ff.close()

In [ ]:
## Opening the full-frame reduction results in code without silas
import h5py

with h5py.File("fullframe_1234501_p78.h5", "r") as f:

    # All point detectors (one value per shot, the rows of the shot table)
    points = f["point_detectors"]

    # Available detectors (labels)
    print(list(points.keys()))

    # Read one detector
    i0 = points["i0_1"][()]

    # Metadata
    print("Detector:", points["i0_1"].attrs["detector"])
    print("Alias   :", points["i0_1"].attrs["alias"])
    print("Unit    :", points["i0_1"].attrs["unit"])

    # Shot metadata
    shots = f["shots"]
    step_index = shots["step_index"][()]
    pump_state = shots["state"][()]               # 0 = on, 1 = off
    used = shots["read"][()].astype(bool)         # the shots in the images

    # Example: the i0_1 values of the pump-on shots of scan step 10
    i0_step10_on = i0[used & (step_index == 10) & (pump_state == 0)]

    # Per step (the rows of step_table), already computed
    i0_mean_on = f["point_steps/i0_1_on_mean"][()]

    # If time-tool binning was on: the shots' bin numbers
    if "bin" in shots:
        bin_index = shots["bin"][()]
        # Example: all i0_1 values in the bin of row 3 of the bin table
        i0_bin3 = i0[used & (bin_index == f["bin_table/bin"][3])]